In [1]:
import praw
import boto3
import os
import random
import requests
import re

from better_profanity import profanity
import emoji
from moviepy import *

from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaFileUpload

profanity.load_censor_words()

MAX_COMMENT_LENGTH = 150

In [2]:
# Setup
polly = boto3.client(
    'polly',
    aws_access_key_id=os.environ.get('AWS_POLLY_ACCESS_KEY'),
    aws_secret_access_key=os.environ.get('AWS_POLLY_SECRET_ACCESS_KEY'),
    region_name='us-west-2',
)
reddit = praw.Reddit(
    client_id=os.getenv("REDDIT_CLIENT_ID"),
    client_secret=os.getenv("REDDIT_CLIENT_SECRET"),
    user_agent=os.getenv("REDDIT_USER_AGENT")
)
subreddit = reddit.subreddit("AskReddit")

In [3]:
# Parse reddit content
with open("prev_post.txt", "r") as f:
    prev_post = f.read().strip()

top_post = next(
    (
        post 
        for post in subreddit.top(time_filter='day', limit=10) 
        if (
            not post.over_18 
            and not profanity.contains_profanity(post.title)
            and not len(post.title) > 90
            and not any(emoji.is_emoji(char) for char in post.title)
            and not post.title == prev_post
        )
    ),
    None
)

if top_post:
    top_post.comment_sort = "top"  # Sort by top comments
    top_post.comments.replace_more(limit=0)  # Replace "load more" placeholders

    # Filter and select top comments within the length limit
    top_comments = [
        comment for comment in top_post.comments 
        if len(comment.body) <= MAX_COMMENT_LENGTH
        and comment.body != "[deleted]"
        and not any(emoji.is_emoji(char) for char in comment.body)
        and not profanity.contains_profanity(comment.body)
    ][:3] 

    # Check if there are enough valid comments
    if len(top_comments) < 3:
        raise ValueError("Not enough valid comments found for video creation.")
else:
    raise ValueError("No suitable Reddit post found.")

if top_post and top_comments:
    reddit_title = top_post.title
    top_comment_0 = top_comments[0].body
    top_comment_1 = top_comments[1].body
    top_comment_2 = top_comments[2].body
    reddit_link = top_post.shortlink

    print("Selected post:", reddit_title)
    print(f"Post link: {reddit_link}")
    print(f"top_comment_0: {top_comment_0}")
    print(f"top_comment_1: {top_comment_1}")
    print(f"top_comment_2: {top_comment_2}")

Selected post: What did you find the most interesting or disturbing in the new Epsteinfiledrop?
Post link: https://redd.it/1qu6f2w
top_comment_0: That victims are not protected by the redaction but the predators are
top_comment_1: That no one really is going to do a thing.
top_comment_2: That Bill Gates, at the time the wealthiest human on earth, had to ask his buddy Epstein for antibiotics.


In [4]:
endpoint_url = f"https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash:generateContent?key={os.environ['GEMINI_API_KEY']}"

headers = {
    "Content-Type": "application/json",
}

prompt = f"""
I'm publishing a YouTube video answering the following question: {reddit_title}.

Here are the responses I'll be highlighting: {top_comment_0}, {top_comment_1}, {top_comment_2}.

Please extract the 10 best topics or keywords I can add to the video to maximize search optimization and virality.
Include a mix of short-tail and long-tail keywords that people searching for answers to this question or related discussions might use. Also, consider terms related to the themes in the provided comments.

Do not include any context or explanations, return only the 10 keywords numbered in the following format:
```
    1. keyword1
    2. keyword2
    3. keyword3
    4. keyword4
    5. keyword5
    6. keyword6
    7. keyword7
    8. keyword8
    9. keyword9
    10.keyword10
```
"""

payload = {
    "contents": [
        {
            "parts": [
                {"text": prompt}
            ]
        }
    ]
}
try:
    response = requests.post(endpoint_url, headers=headers, json=payload)
    candidate_text = response.json()["candidates"][0]["content"]["parts"][0]["text"]
    keywords = re.findall(r'\d+\.\s*(.*)', candidate_text)
    print(keywords)
except Exception as e:
    keywords = []
    

['Epstein file drop', 'Bill Gates Epstein', 'Epstein revelations', 'Epstein victims', 'Predator redactions', 'Epstein scandal', 'Epstein documents', 'Jeffrey Epstein update', 'No accountability Epstein', 'Bill Gates antibiotics']


In [5]:
prompt = f"""
I'm creating a YouTube Short video based on the Reddit question: "{reddit_title}".
The video will feature these key answers from the comments:
1. "{top_comment_0}"
2. "{top_comment_1}"
3. "{top_comment_2}"

Your goal is to craft a highly engaging YouTube video title that maximizes click-through rate (CTR) and encourages virality.

The title should:
- Be a captivating and concise rephrasing or transformation of the original Reddit question: "{reddit_title}".
- Spark strong curiosity and make viewers eager to see the answers.
- Use impactful and engaging language.
- Be suitable for a YouTube Short (generally under 70 characters is good, but impact is key).
- Hint at the nature of the answers/discussion without giving away specifics from the comments.
- NOT be overly clickbaity or sensational.

Important: Return *only* the generated title. Do not include any surrounding quotes, introductory phrases like "Here's a title:", or any other explanatory text. Just the title itself.
"""

payload = {
    "contents": [
        {
            "parts": [
                {"text": prompt}
            ]
        }
    ]
}

try:
    response = requests.post(endpoint_url, headers=headers, json=payload)
    video_title = response.json()["candidates"][0]["content"]["parts"][0]["text"]
    print(video_title)
except Exception as e:
    video_title = None

Epstein Files: What's Truly Disturbing?


In [6]:
# Generate title audio file
voice_id = random.choice(["Danielle", "Stephen"])

try:
    title_resp = polly.synthesize_speech(
        Text=reddit_title,
        OutputFormat='mp3',
        VoiceId=voice_id,
        Engine='neural'
    )

    with open('assets/title_audio.mp3', 'wb') as f:
        f.write(title_resp['AudioStream'].read())
    print(f"Saved title_audio.mp3")

    # Generate comment audio files
    for i, comment in enumerate(top_comments, 1):
        response = polly.synthesize_speech(
            Text=comment.body,
            OutputFormat='mp3',
            VoiceId=voice_id,
            Engine='neural'
        )
        filename = f"assets/comment_{i}.mp3"
        with open(filename, 'wb') as f:
            f.write(response['AudioStream'].read())
        print(f"Saved {filename}")
except Exception as e:
    raise ValueError(f"Failed to generate audio files: {e}.")

Saved title_audio.mp3
Saved assets/comment_1.mp3
Saved assets/comment_2.mp3
Saved assets/comment_3.mp3


In [7]:
# Generate title audio file
voice_id = random.choice(["Danielle", "Stephen"])
text = "Like, subscribe, and comment your answer below!"

try:
    hook = polly.synthesize_speech(
        Text=text,
        OutputFormat='mp3',
        VoiceId="Danielle",
        Engine='neural'
    )

    with open('assets/hook_audio.mp3', 'wb') as f:
        f.write(hook['AudioStream'].read())
    print(f"Saved hook_audio.mp3")
except Exception as e:
    raise ValueError(f"Failed to generate audio files: {e}.")

Saved hook_audio.mp3


In [8]:
def create_segment_clip(bg_img_path, audio_path, text=None, fontsize=60):
    """
    Returns a MoviePy clip of 'background_path' with
    overlay text matching 'audio_path' duration.
    """
    # 1) Load the audio
    audio_clip = AudioFileClip(audio_path)
    audio_duration = audio_clip.duration

    # 2) Load the background img
    bg_clip = ImageClip(bg_img_path)

    # 3) Create a TextClip to overlay
    if text:
        try:
            # For GH action, need dif font
            text_clip = TextClip(
                text=text,
                font_size=fontsize,
                color="#ff5d01",
                # 'caption' mode wraps text automatically to fit the size
                method='caption',
                size=(1000, 1000),
                font='Lato-Bold',
                text_align='center'           
            ).with_duration(audio_duration).with_position(("center", 100))
        except Exception:
            # For local 
            text_clip = TextClip(
                text=text,
                font_size=fontsize,
                color="#ff5d01",
                # 'caption' mode wraps text automatically to fit the size
                method='caption',
                size=(1000, 1000),
                font='Arial',
                text_align='center'           
            ).with_duration(audio_duration).with_position(("center", 100))

        # 4) Combine background + text overlay
        composite = CompositeVideoClip([bg_clip, text_clip]).with_duration(audio_duration)
    else:
        composite = CompositeVideoClip([bg_clip]).with_duration(audio_duration)

    # 5) Set the audio
    composite = composite.with_audio(audio_clip)

    return composite

def create_silent_gap(duration=0.5, background_path="assets/bg.png"):
    """
    Returns a silent color clip (default: 1s black screen).
    """
    return ImageClip(background_path).with_duration(duration)

try:
    if len(reddit_title) < 50:
        fontsize=150
    else:
        fontsize=120
    title_clip = create_segment_clip(
        bg_img_path="assets/bg.png",
        text=reddit_title,
        audio_path="assets/title_audio.mp3",
        fontsize=fontsize
    )

    # Comment clips
    comment_clips = []
    for i, c_text in enumerate([top_comment_0, top_comment_1, top_comment_2], 1):
        if len(c_text) < 50:
            fontsize=100
        elif len(c_text) < 100:
            fontsize=80
        else:
            fontsize=60
        clip = create_segment_clip(
            bg_img_path="assets/bg.png",
            text=c_text,
            audio_path=f"assets/comment_{i}.mp3",
            fontsize=fontsize
        )
        comment_clips.append(clip)

    question_clip = create_segment_clip(
        bg_img_path="assets/bg.png",
        audio_path="assets/question.mp3"
    )

    top_responses_clip = create_segment_clip(
        bg_img_path="assets/bg.png",
        audio_path="assets/top_responses.mp3"
    )

    gap_clip = create_silent_gap()

    hook_clip = create_segment_clip(
        bg_img_path="assets/bg.png",
        audio_path="assets/hook_audio.mp3",
        text="Like, subscribe, and comment your answer below!",
        fontsize=120
    )

    # --- 4. Concatenate all clips (title + comments) into one final video ---
    # Build final sequence: [title, gap, comment 1, gap, comment 2, gap, comment 3]
    all_clips = []
    # all_clips.append(question_clip)
    all_clips.append(gap_clip)
    all_clips.append(title_clip)
    all_clips.append(gap_clip)
    all_clips.append(top_responses_clip)
    all_clips.append(gap_clip)

    for comment_clip in comment_clips:
        all_clips.append(comment_clip)
        all_clips.append(gap_clip)

    all_clips.append(hook_clip)
    final_clip = concatenate_videoclips(all_clips, method="compose")

    # --- 5. Add background music ---
    background_music = AudioFileClip("assets/funk_bg_lower.mp3").with_duration(final_clip.duration)
    composite_audio = CompositeAudioClip([final_clip.audio, background_music])
    final_clip = final_clip.with_audio(composite_audio)

    # --- 6. Export the final video ---
    final_clip.write_videofile("assets/final_askreddit_video.mp4", fps=30)
    
    thumbnail_path = "assets/thumbnail.png"
    title_clip.save_frame(thumbnail_path, t=0)

except Exception as e:
    raise ValueError(f"Failed to generate video clips: {e}.")


MoviePy - Building video assets/final_askreddit_video.mp4.
MoviePy - Writing audio in final_askreddit_videoTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video assets/final_askreddit_video.mp4



frame_index:  79%|███████▉  | 671/852 [04:15<01:12,  2.48it/s, now=None]

KeyboardInterrupt: 

In [ ]:
def authenticate_youtube():
    creds = Credentials(
        None,
        refresh_token=os.environ['YOUTUBE_REFRESH_TOKEN'],
        token_uri="https://oauth2.googleapis.com/token",
        client_id=os.environ['YOUTUBE_CLIENT_ID'],
        client_secret=os.environ['YOUTUBE_CLIENT_SECRET']
    )

    # Refresh the token if necessary
    if not creds.valid:
        creds.refresh(Request())
    
    return creds

# --- 8. Upload to YouTube ---
def upload_to_youtube(file_path, title, description, category_id="22", keywords=None, privacy_status="public"):
    """
    Uploads a video to YouTube.

    :param file_path: Path to the video file.
    :param title: Title of the video.
    :param description: Description of the video.
    :param category_id: YouTube video category ID (default is "People & Blogs").
    :param keywords: List of keywords/tags.
    :param privacy_status: "public", "private", or "unlisted".
    """
    creds = authenticate_youtube()
    youtube = build('youtube', 'v3', credentials=creds)

    try:
        body = {
            'snippet': {
                'title': title,
                'description': description,
                'tags': keywords if keywords else [],
                'categoryId': category_id
            },
            'status': {
                'privacyStatus': privacy_status
            }
        }

        # Upload the video file
        media_body = MediaFileUpload(file_path, chunksize=-1, resumable=True, mimetype='video/mp4')

        request = youtube.videos().insert(
            part=','.join(body.keys()),
            body=body,
            media_body=media_body
        )

        response = None
        while response is None:
            status, response = request.next_chunk()
            if status:
                print(f"Uploading... {int(status.progress() * 100)}%")

        print(f"Upload Complete! Video ID: {response['id']}")
        return response['id']

    except HttpError as e:
        print(f"An HTTP error {e.resp.status} occurred:\n{e.content}")
        return None

def upload_thumbnail(video_id, thumbnail_path):
    """
    Uploads a thumbnail image for the given video ID.
    """
    creds = authenticate_youtube()
    youtube = build('youtube', 'v3', credentials=creds)
    try:
        thumbnail_request = youtube.thumbnails().set(
            videoId=video_id,
            media_body=MediaFileUpload(thumbnail_path, mimetype='image/jpeg')
        )
        response = thumbnail_request.execute()
        print("Thumbnail uploaded:", response)
    except HttpError as e:
        print(f"Failed to upload thumbnail: {e}")
        
try:
    # Define video metadata
    video_title = video_title if video_title else reddit_title
    video_title = f"{video_title} #shorts #foryou"
    video_description = (
        f"Today's top AskReddit post: {reddit_title}\n\n"
        f"Top Comments:\n1. {top_comment_0}\n2. {top_comment_1}\n3. {top_comment_2}\n\n"
        f"{reddit_link}\n"
        "#AskReddit #RedditDaily #shorts #foryou"
    )

    video_keywords = [
        "AskReddit",
        "Ask Reddit",
        "Shorts", 
        "Reddit",
        "Top AskReddit Post",
        "Trending AskReddit",
    ] + keywords

    # Upload the video
    video_id = upload_to_youtube(
        file_path="assets/final_askreddit_video.mp4",
        title=video_title,
        description=video_description,
        category_id="22",  # People & Blogs
        keywords=video_keywords,
        privacy_status="public" 
    )
    if video_id:
        print(f"Video successfully uploaded! Watch it at https://www.youtube.com/watch?v={video_id}")
        upload_thumbnail(video_id, thumbnail_path)
except Exception as e:
    raise ValueError(f"Failed upload. {e}")

ValueError: Failed upload. ('invalid_grant: Bad Request', {'error': 'invalid_grant', 'error_description': 'Bad Request'})

In [ ]:
if top_post:
    with open("prev_post.txt", "w") as f:
        f.write(top_post.title)